In [1]:
import os, sys, re, math
from pathlib import Path

import torch, dgl  # noqa: F401
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ajouter le repo au PYTHONPATH
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

from python.create_dgl_dataset import TelemacDataset
from python.CustomMeshGraphNet import MeshGraphNet
from modulus.launch.utils import load_checkpoint

import math
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [17]:
DATA_DIR = #"/work/m24046/m24046mrcr/paper/Experience1/Mesh8_base.bin"#"/work/m24046/m24046mrcr/paper/Experience2/Multimesh_8_32.bin"

DYNAMIC_DIR = [ 
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_2600_Group_1_peak_2600_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_2_peak_1000_Group_2_peak_1000_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_2_peak_1200_Group_2_peak_1200_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_2_peak_1600_Group_2_peak_1600_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_4_peak_2000_Group_4_peak_2000_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_1200_Group_1_peak_1200_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_2400_Group_1_peak_2400_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_3_peak_3400_Group_3_peak_3400_0_0-80_interpolated.pkl",
]

EXPERIMENTS = {
     "E2_Baseline": {"ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience2/Seed0/", "epochs": [600, 700, 800, 850, 900]},
     "E4_pushforward": {"ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience4_1/Seed0/", "epochs": [600, 700, 800, 850, 900]},
     "E6_1_pushforward_tversky_07_03": {"ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience6_1/Seed0/", "epochs": [600, 700, 800, 850, 900,1000]},
     "E6_2_pushforward_tversky_05_05": {"ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience6_2/Seed0/", "epochs": [600, 700, 800, 850,900,1000]},
     "E3_pushforward": {"ckpt_dir": "/work/m24046/m24046mrcr/paper/Experience3_1/Seed0/", "epochs": [600, 700, 800, 850, 900]},
}

NUM_INPUT_FEATURES = 9
NUM_EDGE_FEATURES = 3
NUM_OUTPUT_FEATURES = 3
MP_LAYERS = 10
DO_CONCAT_TRICK = True
NUM_PROCESSOR_CHECKPOINT_SEGMENTS = 0

# Horizons (pas de 30 min)
HORIZONS_STEPS = [3,6, 12, 24]
THRESHOLD_M = 0.05

In [18]:
# =====================
# Modèle / dataset helpers
# =====================

def build_model():
    return MeshGraphNet(
        NUM_INPUT_FEATURES,
        NUM_EDGE_FEATURES,
        NUM_OUTPUT_FEATURES,
        processor_size=MP_LAYERS,
        hidden_dim_processor=64,
        hidden_dim_node_encoder=64,
        hidden_dim_edge_encoder=64,
        hidden_dim_node_decoder=64,
        do_concat_trick=DO_CONCAT_TRICK,
        num_processor_checkpoint_segments=NUM_PROCESSOR_CHECKPOINT_SEGMENTS,
    )

def load_model_checkpoint(model, ckpt_dir, epoch):
    load_checkpoint(ckpt_dir, models=model, device=device, epoch=epoch)
    model.to(device)
    model.eval()
    return model

def build_dataset_for_event(dynamic_file, ckpt_dir, sequence_length, overlap, split="test"):
    return TelemacDataset(
        name=f"eval_{split}",
        data_dir=DATA_DIR,
        dynamic_data_files=[dynamic_file],
        split=split,
        ckpt_path=ckpt_dir,
        normalize=True,
        sequence_length=sequence_length,
        overlap=overlap,
    )

def _denorm(xn, mean, std):
    return xn * std + mean

def _renorm(x, mean, std):
    return (x - mean) / (std + 1e-12)


In [19]:
# -----------------------
# Paramètres
# -----------------------
TEST_EXP = list(EXPERIMENTS.keys())[1]
TEST_EPOCH = 900
TEST_EVENT = DYNAMIC_DIR[1]
SPLIT = "test"

THR = 0.01  # seuil wet/dry pour CSI et ROI "initial wet"

SEQ_LEN = max(HORIZONS_STEPS) + 1
OVERLAP_TEST = SEQ_LEN - 1
MAX_SEQ = None

ckpt_dir = EXPERIMENTS[TEST_EXP]["ckpt_dir"]
ds = build_dataset_for_event(TEST_EVENT, ckpt_dir, sequence_length=SEQ_LEN, overlap=OVERLAP_TEST, split=SPLIT)

model = build_model()
model = load_model_checkpoint(model, ckpt_dir, epoch=TEST_EPOCH)

#
# Test par distance de HOP
#

rows_band = []

bands = [(0,5), (6,10), (11,15), (16,20), (21,30)]

# -----------------------
# Normalisation
# -----------------------
stats = ds.node_stats
dyn_start = ds.base_graph.ndata["static"].shape[1]

mx = torch.tensor([stats["h"].item(), stats["u"].item(), stats["v"].item()], device=device)
sx = torch.tensor([stats["h_std"].item(), stats["u_std"].item(), stats["v_std"].item()], device=device)
dy_mean = torch.tensor([stats["delta_h"].item(), stats["delta_u"].item(), stats["delta_v"].item()], device=device)
dy_std = torch.tensor([stats["delta_h_std"].item(), stats["delta_u_std"].item(), stats["delta_v_std"].item()], device=device)

# -----------------------
# Métriques
# -----------------------
def csi_masked(h_pred, h_gt, mask, thr):
    if mask.sum().item() == 0:
        return float("nan")
    hp = h_pred[mask] >= thr
    hg = h_gt[mask] >= thr
    tp = (hp & hg).sum().item()
    fp = (hp & (~hg)).sum().item()
    fn = ((~hp) & hg).sum().item()
    denom = tp + fp + fn
    return tp / denom if denom > 0 else float("nan")

def wet_frac(h, mask, thr):
    if mask.sum().item() == 0:
        return float("nan")
    return (h[mask] >= thr).float().mean().item()

# -----------------------
# Rollout
# -----------------------
def rollout_to_horizons(graphs, q_mask, h_mask, bc_mode, fixed_q=None, fixed_h=None):
    g = graphs[0].to(device)
    static_part = g.ndata["x"][:, :dyn_start]
    xn_t = g.ndata["x"][:, dyn_start:dyn_start + 3]

    captured = {}
    max_h = max(HORIZONS_STEPS)

    for t in range(max_h):
        with torch.no_grad():
            y_pred_n = model(g.ndata["x"], g.edata["x"], g)

        x_t  = _denorm(xn_t, mx, sx)
        dy   = _denorm(y_pred_n, dy_mean, dy_std)
        x_t1 = x_t + dy

        x_gt_n = graphs[t + 1].ndata["x"][:, dyn_start:dyn_start + 3].to(device)
        x_gt   = _denorm(x_gt_n, mx, sx)

        if bc_mode == "gt":
            x_t1[q_mask] = x_gt[q_mask]
            x_t1[h_mask, 0:1] = x_gt[h_mask, 0:1]
        elif bc_mode == "fixed":
            x_t1[q_mask] = fixed_q
            x_t1[h_mask, 0:1] = fixed_h
        else:
            raise ValueError(bc_mode)

        step = t + 1
        if step in HORIZONS_STEPS:
            captured[step] = (x_t1.detach().clone(), x_gt.detach().clone())

        xn_t = _renorm(x_t1, mx, sx)
        g = g.clone()
        g.ndata["x"] = torch.cat([static_part, xn_t], dim=1)

    return captured

# -----------------------
# Boucle principale
# -----------------------
rows = []
nseq = len(ds) if MAX_SEQ is None else min(MAX_SEQ, len(ds))

for seq_idx in range(nseq):
    graphs = ds[seq_idx]
    if len(graphs) <= max(HORIZONS_STEPS):
        continue

    g0 = graphs[0].to(device)

    # Masques BC à partir du one-hot
    static_part = g0.ndata["x"][:, :dyn_start]
    onehot = static_part[:, :4]
    q_mask = (onehot == torch.tensor([0, 0, 1, 0], device=device)).all(dim=1)
    h_mask = (onehot == torch.tensor([0, 1, 0, 0], device=device)).all(dim=1)

    # ROI = zone initialement sèche (t0)
    x0_n = g0.ndata["x"][:, dyn_start:dyn_start + 3]
    x0 = _denorm(x0_n, mx, sx)
    roi = x0[:, 0] < THR

    # (Optionnel mais souvent souhaitable) exclure les BC de la ROI
    roi = roi & ~(q_mask | h_mask)
    
    dist = compute_hop_distance_from_bc(g0, q_mask)


    # BC fixées à t0 (unités physiques)
    fixed_q = x0[q_mask].clone()
    fixed_h = x0[h_mask, 0:1].clone()

    cap_gt = rollout_to_horizons(graphs, q_mask, h_mask, bc_mode="gt")
    cap_fx = rollout_to_horizons(graphs, q_mask, h_mask, bc_mode="fixed", fixed_q=fixed_q, fixed_h=fixed_h)

    for h in HORIZONS_STEPS:
        if h not in cap_gt or h not in cap_fx:
            continue

        pred_gt, gt = cap_gt[h]
        pred_fx, _  = cap_fx[h]

        # Effet "BC fixées" vs "BC GT"
        delta_h_roi = (pred_fx[:, 0] - pred_gt[:, 0]).abs()[roi].mean().item() if roi.any() else float("nan")

        # Skill vs GT
        csi_gt = csi_masked(pred_gt[:, 0], gt[:, 0], roi, THR)
        csi_fx = csi_masked(pred_fx[:, 0], gt[:, 0], roi, THR)

        rows.append({
            "seq_idx": seq_idx,
            "horizon": h,
            "roi_count": int(roi.sum().item()),
            "delta_h_roi_mae": delta_h_roi,
            "csi_gtBC_roi": csi_gt,
            "csi_fixedBC_roi": csi_fx,
            "delta_csi_roi": (csi_fx - csi_gt) if (not math.isnan(csi_fx) and not math.isnan(csi_gt)) else float("nan"),
            "wet_gt_roi": wet_frac(gt[:, 0], roi, THR),
            "wet_pred_gtBC_roi": wet_frac(pred_gt[:, 0], roi, THR),
            "wet_pred_fixedBC_roi": wet_frac(pred_fx[:, 0], roi, THR),
        })
        
        for dmin, dmax in bands:
            band_mask = (dist >= dmin) & (dist <= dmax)
            band_mask = band_mask.to(device)

            band_roi = roi & band_mask

            if band_roi.sum() == 0:
                continue

            csi_gt_band = csi_masked(pred_gt[:,0], gt[:,0], band_roi, THR)
            csi_fx_band = csi_masked(pred_fx[:,0], gt[:,0], band_roi, THR)

            rows_band.append({
                "seq_idx": seq_idx,
                "horizon": h,
                "dmin": dmin,
                "dmax": dmax,
                "count": int(band_roi.sum().item()),
                "csi_gt": csi_gt_band,
                "csi_fx": csi_fx_band,
                "delta_csi": csi_fx_band - csi_gt_band
            })


df = pd.DataFrame(rows)
display(df.head(10))

summary = df.groupby("horizon").agg(
    n=("seq_idx", "count"),
    roi_count_mean=("roi_count", "mean"),
    delta_h_roi_mae_mean=("delta_h_roi_mae", "mean"),
    delta_h_roi_mae_std=("delta_h_roi_mae", "std"),
    csi_gtBC_roi_mean=("csi_gtBC_roi", "mean"),
    csi_fixedBC_roi_mean=("csi_fixedBC_roi", "mean"),
    delta_csi_roi_mean=("delta_csi_roi", "mean"),
).reset_index()

display(summary)

[10:13:03 - checkpoint - INFO] Loaded model state dictionary /work/m24046/m24046mrcr/paper/Experience4_1/Seed0/MeshGraphNet.0.900.mdlus to device cuda
[10:13:03 - checkpoint - INFO] Loaded checkpoint file /work/m24046/m24046mrcr/paper/Experience4_1/Seed0/checkpoint.0.900.pt to device cuda


Loading normalization statistics...


,seq_idx,horizon,roi_count,delta_h_roi_mae,csi_gtBC_roi,csi_fixedBC_roi,delta_csi_roi,wet_gt_roi,wet_pred_gtBC_roi,wet_pred_fixedBC_roi
0,0,3,14127,1.127151e-06,0.150794,0.151194,0.000400,0.007857,0.022935,0.022864
1,0,6,14127,1.094124e-05,0.093361,0.093296,-0.000065,0.016776,0.095137,0.095208
2,0,12,14127,8.966745e-05,0.067157,0.067267,0.000109,0.069371,0.854109,0.853826
3,0,24,14127,1.583257e-03,0.100595,0.100375,-0.000220,0.103773,0.956466,0.956254
4,1,3,14084,8.466496e-07,0.112329,0.112329,0.000000,0.007881,0.020946,0.020946
5,1,6,14084,8.844010e-06,0.077085,0.077031,-0.000054,0.017893,0.091238,0.091309
6,1,12,14084,8.221465e-05,0.084014,0.084117,0.000102,0.086836,0.854942,0.854800
7,1,24,14084,1.159995e-03,0.094901,0.095090,0.000190,0.097203,0.956262,0.955978
8,2,3,14054,9.701264e-07,0.096515,0.096515,0.000000,0.008610,0.020492,0.020492
9,2,6,14054,9.412409e-06,0.072474,0.071777,-0.000697,0.019781,0.089725,0.089654


,horizon,n,roi_count_mean,delta_h_roi_mae_mean,delta_h_roi_mae_std,csi_gtBC_roi_mean,csi_fixedBC_roi_mean,delta_csi_roi_mean
0,3,7,14011.428571,0.000002,0.000002,0.098606,0.098430,-0.000176
1,6,7,14011.428571,0.000031,0.000021,0.073419,0.073036,-0.000383
2,12,7,14011.428571,0.000112,0.000021,0.095548,0.095584,0.000036
3,24,7,14011.428571,0.000985,0.000315,0.089504,0.089472,-0.000032


In [20]:
df_band = pd.DataFrame(rows_band)

summary_band = df_band.groupby(
    ["horizon", "dmin", "dmax"]
).agg(
    n=("seq_idx", "count"),
    count_mean=("count", "mean"),
    csi_gt_mean=("csi_gt", "mean"),
    csi_fx_mean=("csi_fx", "mean"),
    delta_csi_mean=("delta_csi", "mean"),
).reset_index()

display(summary_band)


,horizon,dmin,dmax,n,count_mean,csi_gt_mean,csi_fx_mean,delta_csi_mean
0,3,0,5,7,98.714286,0.305102,0.302381,-2.721088e-03
1,3,6,10,7,163.714286,0.323182,0.323182,0.000000e+00
2,3,11,15,7,319.857143,0.286905,0.286905,0.000000e+00
3,3,16,20,7,546.571429,0.350600,0.350600,0.000000e+00
4,3,21,30,7,1691.571429,0.150511,0.150511,0.000000e+00
5,6,0,5,7,98.714286,0.448100,0.423243,-2.485689e-02
6,6,6,10,7,163.714286,0.187703,0.187703,0.000000e+00
7,6,11,15,7,319.857143,0.246485,0.246485,0.000000e+00
8,6,16,20,7,546.571429,0.459467,0.459467,0.000000e+00
9,6,21,30,7,1691.571429,0.151678,0.151678,0.000000e+00


In [21]:
from collections import deque

def compute_hop_distance_from_bc(g, bc_mask):
    """
    g : DGLGraph (sur CPU)
    bc_mask : bool tensor (num_nodes,) indiquant les nœuds BC
    retourne : tensor (num_nodes,) des distances en hops
    """
    g = g.to("cpu")
    num_nodes = g.num_nodes()
    
    # adjacency list
    src, dst = g.edges()
    src = src.numpy()
    dst = dst.numpy()
    
    adj = [[] for _ in range(num_nodes)]
    for s, d in zip(src, dst):
        adj[s].append(d)
        adj[d].append(s)  # graphe non orienté

    # initialisation distances
    dist = torch.full((num_nodes,), float("inf"))
    
    queue = deque()
    
    # multi-source initialisation
    for i in torch.where(bc_mask.cpu())[0]:
        dist[i] = 0
        queue.append(int(i))
    
    # BFS
    while queue:
        u = queue.popleft()
        for v in adj[u]:
            if dist[v] == float("inf"):
                dist[v] = dist[u] + 1
                queue.append(v)
    
    return dist


In [22]:
g0 = graphs[0].to(device)

# BC mask
static_part = g0.ndata["x"][:, :dyn_start]
onehot = static_part[:, :4]
q_mask = (onehot == torch.tensor([0,0,1,0], device=device)).all(dim=1)
#h_mask = (onehot == torch.tensor([0,1,0,0], device=device)).all(dim=1)
bc_mask = q_mask 

# ROI extent (zone initialement sèche)
x0_n = g0.ndata["x"][:, dyn_start:dyn_start+3]
x0 = _denorm(x0_n, mx, sx)
roi = (x0[:,0] < THR) & ~(bc_mask)

# calcul distances
dist = compute_hop_distance_from_bc(g0, bc_mask)

# statistiques sur la ROI
roi_dist = dist[roi.cpu()]

print("Distance moyenne BC → ROI :", roi_dist.mean().item())
print("Distance médiane :", roi_dist.median().item())
print("Distance max :", roi_dist.max().item())


Distance moyenne BC → ROI : 45.85320281982422
Distance médiane : 48.0
Distance max : 78.0
